In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *

In [89]:
df = pd.read_excel('app1.xlsx')
df.set_index('maquinas',inplace=True)
df.T

maquinas,fresa,tmecanico,tautomatico
peca1,10,20,30
peca2,20,30,80


In [ ]:
df['peca1']

np.int64(10)

In [92]:
peças = list(df.columns)
maquinas = list(df.index)

quantidade = {
    maquinas[0]:3,
    maquinas[1]:3,
    maquinas[2]:1,
}



In [132]:
model  = pyo.ConcreteModel()

model.pecas = pyo.Set(initialize = peças)
model.maquinas = pyo.Set(initialize = maquinas)

# variavel x criada para indicar o tempo de cada maquina em cada peça, nao sei se é isso que querem
model.x = pyo.Var(model.maquinas,model.pecas, bounds=(0,None), domain=pyo.NonNegativeReals)

model.z = pyo.Var(bounds=(0, None), domain=pyo.NonNegativeReals)

# objetivo: maximizar produtos completos
def obj(model):
    return model.z
model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

# z é limitado pelo total de peca1 produzida
def lim_peca1(model):
    return model.z <= sum(model.x[m,'peca1'] * quantidade[m] * df['peca1'][m] for m in model.maquinas)
model.lim1 = pyo.Constraint(rule=lim_peca1)

# z é limitado pelo total de peca2 produzida
def lim_peca2(model):
    return model.z <= sum(model.x[m,'peca2'] * quantidade[m] * df['peca2'][m] for m in model.maquinas)
model.lim2 = pyo.Constraint(rule=lim_peca2)
# pensando em criar uma variavel para quantidade de peca produzida
# def obj(model):
#     return sum(model.x[m,p] *quantidade[m] * df[p][m] for m in model.maquinas for p in model.pecas )
# model.obj=pyo.Objective(rule=obj, sense=pyo.maximize)

# restricao_tempo
def tempo(model,m):
    return sum(model.x[m,p] for p in model.pecas) == 1
model.tempo = pyo.Constraint(model.maquinas, rule=tempo)


In [133]:
model.pprint()

2 Set Declarations
    maquinas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'fresa', 'tmecanico', 'tautomatico'}
    pecas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'peca1', 'peca2'}

2 Var Declarations
    x : Size=6, Index=maquinas*pecas
        Key                      : Lower : Value : Upper : Fixed : Stale : Domain
              ('fresa', 'peca1') :     0 :  None :  None : False :  True : NonNegativeReals
              ('fresa', 'peca2') :     0 :  None :  None : False :  True : NonNegativeReals
        ('tautomatico', 'peca1') :     0 :  None :  None : False :  True : NonNegativeReals
        ('tautomatico', 'peca2') :     0 :  None :  None : False :  True : NonNegativeReals
          ('tmecanico', 'peca1') :     0 :  None :  None : False :  True : NonNegativeReals
          ('tmecanico', 'peca2') :     0 :  None :

In [134]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp0qu8weor.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpes0d48ul.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpes0d48ul.pyomo.lp
Objective sense      : Maximize
Variables            :       7
Objective nonzeros   :       1
Linear constraints   :       5  [Less: 2,  Equal: 3]
  Nonzeros           :      14
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 1.000000         Max   : 1

In [135]:
for m,p in model.x:
    print(f"{m,p}: {pyo.value(model.x[m,p])}")

('fresa', 'peca1'): 0.8888888888888893
('fresa', 'peca2'): 0.11111111111111072
('tmecanico', 'peca1'): 1.0
('tmecanico', 'peca2'): 0.0
('tautomatico', 'peca1'): 0.0
('tautomatico', 'peca2'): 1.0
